In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from time import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer



## Passo 1: Obter conjunto de datasets de treinamento

In [ ]:
#Carregando os datasets
from sklearn.datasets import fetch_openml

names = ['diabetes', 'blood-transfusion-service-center',
         'monks-problems-2', 'tic-tac-toe', 'titanic', 'pc1',
         'kr-vs-kp', 'phoneme', 'wdbc', 'semeion', 'isolet',
         'cnae-9', 'ilpd-numeric', 'students_scores', 'kits',
         'usps', 'ibm-employee-performance','mushroom',
         'segment',  'autoUniv-au1-1000', 'pizzacutter3', 
         'qsar', 'solar-flare']

#errors: 'sick-numeric', 'telco-custumer-churn', 'credit-g', 'anneal'

datasets = {} 
for name in names:
    print('Fetching dataset: {}'.format(name))
    datasets[name] = fetch_openml(name=name, as_frame=True)

print(f'Finished fetching {len(datasets)} datasets.')


## Passo 2: Avaliar performance dos classificadores nos datasets

In [9]:
# Define classifiers
from sklearn.impute import SimpleImputer


classifiers = {
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'Perceptron': Perceptron(random_state=42, max_iter=1000),
    'MLP': MLPClassifier(random_state=42, max_iter=1000)
}

# Store results
results = []

# Iterate through datasets
for dataset_name in names:
    print(f'Processing dataset: {dataset_name}')
    
    # Get data and target
    X = datasets[dataset_name]['data']
    y = datasets[dataset_name]['target']
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        print(f'Warning: Imputation failed for {dataset_name} with error: {e}')
        print('Falling back to simple imputation strategy.')
        # Use most frequent strategy for string/categorical data
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        # Fit and transform the data
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    # Evaluate each classifier
    for clf_name, clf in classifiers.items():
        print(f'  Evaluating {clf_name}...', end=' ')
        
        # 5-fold cross validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_results = cross_validate(clf, X, y, cv=cv, scoring='accuracy', 
                                    return_train_score=False)
        
        # Extract fold accuracies
        fold_accs = cv_results['test_score']
        
        # Create result row
        result_row = {
            'Dataset': dataset_name,
            'Classifier': clf_name,
            'acc_fold1': fold_accs[0],
            'acc_fold2': fold_accs[1],
            'acc_fold3': fold_accs[2],
            'acc_fold4': fold_accs[3],
            'acc_fold5': fold_accs[4],
            'acc_mean': fold_accs.mean(),
            'acc_stddev': fold_accs.std(),
            'train_time': cv_results['fit_time'].sum(),
            'test_time': cv_results['score_time'].sum()
        }
        
        results.append(result_row)
        print('Done')

# Create results DataFrame
performances_df = pd.DataFrame(results)

Processing dataset: diabetes
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: blood-transfusion-service-center
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: monks-problems-2
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... 

/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: Conv

Done
Processing dataset: tic-tac-toe
Falling back to simple imputation strategy.
  Evaluating DecisionTree... 

/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/meta_learning_course/lib/python3.14/site-packages/sklearn/tree/_classes.py", line 1027, in fit
    super()._fit(
    ~~~~~~~~~~~~^
      

In [18]:
performances_df.head(10)

,Dataset,Classifier,acc_fold1,acc_fold2,acc_fold3,acc_fold4,acc_fold5,acc_mean,acc_stddev,train_time,test_time
0,diabetes,DecisionTree,0.727273,0.733766,0.707792,0.712418,0.620915,0.700433,0.040871,0.019557,0.003930
1,diabetes,SVM,0.759740,0.805195,0.753247,0.732026,0.751634,0.760368,0.024249,0.031062,0.015027
2,diabetes,KNN,0.714286,0.753247,0.662338,0.692810,0.666667,0.697869,0.033460,0.006960,0.007045
3,diabetes,LogisticRegression,0.779221,0.779221,0.785714,0.764706,0.745098,0.770792,0.014574,0.103822,0.007497
4,diabetes,Perceptron,0.616883,0.629870,0.623377,0.352941,0.653595,0.575333,0.111886,0.013662,0.009161
5,diabetes,MLP,0.571429,0.668831,0.681818,0.653595,0.718954,0.658925,0.048805,0.316391,0.007636
6,blood-transfusion-service-center,DecisionTree,0.680000,0.740000,0.740000,0.711409,0.677852,0.709852,0.027333,0.006402,0.003018
7,blood-transfusion-service-center,SVM,0.766667,0.753333,0.753333,0.771812,0.751678,0.759365,0.008247,0.027740,0.011373
8,blood-transfusion-service-center,KNN,0.713333,0.793333,0.760000,0.812081,0.731544,0.762058,0.036851,0.003419,0.005336
9,blood-transfusion-service-center,LogisticRegression,0.800000,0.760000,0.773333,0.785235,0.744966,0.772707,0.019148,0.040311,0.003215


## Passo 3: Extrair meta-features dos datasets

In [ ]:
# Install and extract meta-features using pymfe
try:
    from pymfe.mfe import MFE
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'pymfe'])
    from pymfe.mfe import MFE

# Extract meta-features
meta_features_results = []

for dataset_name in names:
    print(f'Extracting meta-features from {dataset_name}...', end=' ')
    
    # Get data and target
    X = datasets[dataset_name]['data']
    y = datasets[dataset_name]['target']
    
    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    # Extract meta-features
    try:
        mfe = MFE(groups=["statistical", "info-theory", "model-based", "landmarking"])
        mfe.fit(X.values, y.values)
        meta_feature_dict = mfe.extract()
        
        # Create result row with dataset name and meta-features
        result_row = {'dataset': dataset_name}
        for feature_name, feature_value in zip(meta_feature_dict[0], meta_feature_dict[1]):
            result_row[feature_name] = feature_value
        
        meta_features_results.append(result_row)
        print('Done')
    except Exception as e:
        print(f'Error: {e}')

# Create meta-features DataFrame
meta_features_df = pd.DataFrame(meta_features_results)

In [ ]:
print('\nMeta-features DataFrame:')
print(meta_features_df.head())
print(f'\nShape: {meta_features_df.shape}')
print(f'\nColumns: {meta_features_df.columns.tolist()}')